[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLManuals/Deep-Learning/blob/main/chapter07_backpropagation_from_the_ground_up/notebook.ipynb)

# Chapter 7: Backpropagation From the Ground Up

Run the cell below once, first, to install any missing packages (only needed on Google Colab; skip it if you're running locally and already have them).

In [ ]:
%pip install -q torch numpy scikit-learn

In October 1986, the journal *Nature* published a four page paper by David Rumelhart, Geoffrey Hinton and Ronald Williams called "Learning representations by back-propagating errors". It described a way for a network with hidden layers to work out how every one of its weights should change, and it is the paper usually credited with bringing neural networks back from the dead after the long winter that followed Minsky and Papert's book in Chapter 3.

The curious thing is that the method was not new. Seppo Linnainmaa, a Finnish master's student, had written down the same calculation in 1970, though not for neural networks. Paul Werbos had spelled out how to use it to train networks in his Harvard doctoral thesis in 1974, and had largely kept it to himself through the winter years. Others arrived at it independently around the same time as the 1986 paper. The idea kept being discovered, and kept failing to catch on.

What the 1986 paper added was a clear demonstration of why it mattered. The authors pointed out that the real prize was the network's ability to invent useful new features in its hidden layers, which was exactly what the Perceptron of Chapter 2 could never do. That was the argument that stuck, and the method now has a name every practitioner knows, **backpropagation**, usually shortened to backprop.

Chapter 6 left one piece missing. Gradient descent needs the slope of the loss for every weight, and measuring slopes by nudging costs two guesses per weight, which is hopeless for a real network. Backprop is how you get all of those slopes in one sweep. Before you read on, try to guess the answers to these four questions.

1. **If a change in A causes a change in B, which causes a change in C, how do you work out A's effect on C?**
2. **A hidden neuron never sees the right answer, so how can it know whether it made things worse?**
3. **Why is the backward direction the natural one for this calculation?**
4. **Roughly how much more work is one backward sweep than one forward pass?**

## The Chain Rule, Gently

Everything in backprop rests on one idea that you already use without naming it. Imagine a bakery where the price of flour affects the cost of a loaf, and the cost of a loaf affects the day's profit. Suppose that adding one rupee to the price of flour adds three rupees to the cost of a loaf, and that adding one rupee to the cost of a loaf removes two rupees from the profit. What does one rupee on flour do to the profit?

You multiply. One rupee on flour becomes three rupees on cost, and each of those three costs two rupees of profit, so the profit falls by six. That is the **chain rule**, and it is nothing more than the common sense that effects multiply as they pass along a chain.

**To find how much a change at the start of a chain affects the end, multiply the effects of every link along the way.**

It works the same whether a chain has two links or two hundred, as Figure 7.1 shows. The links are multiplied in order, and the answer is the overall effect from one end to the other.

> 🖼️ **Figure 7.1:** The chain rule. Each link has its own effect, and the effect from one end of the chain to the other is their product.

A neural network is exactly such a chain. A weight in the first layer changes that layer's weighted sum, which changes its activation, which changes the next layer's weighted sum, and so on until it changes the loss. To find how that one weight affects the loss, you multiply the effects along the chain. Backprop is this done carefully, and done for every weight at once.

## Passing the Error Backward

Take the smallest network that has something to teach, with two inputs, two hidden neurons and one output, all using the sigmoid from Chapter 4. The next cell sets it up with weights chosen to keep the arithmetic clean, and feeds it the input 1 and 0 with a target answer of 1.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

W1 = np.array([[0.5, -0.5],
               [0.5,  0.5]])
b1 = np.array([0.0, 0.0])
W2 = np.array([1.0, -1.0])
b2 = 0.0

x = np.array([1.0, 0.0])
target = 1.0

The forward pass is the one from Chapter 3, now with a loss at the end. Each hidden neuron takes its weighted sum and bends it, the output neuron does the same, and the squared error from Chapter 5 scores the result.

In [ ]:
z1 = W1 @ x + b1
h = sigmoid(z1)
z2 = W2 @ h + b2
y = sigmoid(z2)
loss = (y - target) ** 2
print(z1.round(4), h.round(4))
print(round(z2, 4), round(y, 4), round(loss, 4))

Both hidden neurons produce 0.6225, and because the output weights are $+1$ and $-1$ they cancel exactly, leaving a weighted sum of zero and an output of 0.5. The network is completely undecided, and the loss is 0.25.

Now walk back along the chain. Each step asks the same question, which is how much the loss changes when this quantity changes, and each answer is built from the one before it.

The first link is the loss itself. The squared error $(y - \text{target})^2$ changes at a rate of $2(y - \text{target})$, which is $2 \times (0.5 - 1) = -1$. The next link is the output neuron's bend. Chapter 4 gave the sigmoid's slope as its output times one minus its output, which here is $0.5 \times 0.5 = 0.25$. Multiplying the two links gives how much the loss changes with the output's weighted sum.

In [ ]:
dz2 = 2 * (y - target) * y * (1 - y)
print(round(dz2, 4))

That single number, often called the output's **error signal**, is the key to everything that follows, because every weight feeding into this neuron shares it. The weighted sum is $W_2 \cdot h + b_2$, so raising a weight by one unit raises the sum by whatever that weight was multiplied by, namely its hidden neuron's output. The bias is multiplied by nothing, so it simply inherits the error signal.

In [ ]:
grad_W2 = dz2 * h
grad_b2 = dz2
print(grad_W2.round(4), round(grad_b2, 4))

Each weight's gradient is the shared error signal scaled by the hidden output it was multiplied by, while the bias, multiplied by nothing, simply inherits the error signal unchanged.

Now comes the step that gives backprop its name and that puzzled researchers for years. A hidden neuron never sees the target, so how can it know what it did wrong? The answer is that the output neuron tells it. The hidden neuron's influence on the loss travels through the weight joining them, so each hidden neuron receives the output's error signal multiplied by its own outgoing weight. A neuron connected by a large weight carries more of the blame, and one connected by a negative weight gets the blame with the sign flipped.

In [ ]:
dh = dz2 * W2
dz1 = dh * h * (1 - h)
print(dh.round(4), dz1.round(4))

Each hidden neuron now has its own error signal, and these are used exactly as the output's was. The gradients for the first layer's weights are each neuron's error signal multiplied by the input it was applied to.

In [ ]:
grad_W1 = np.outer(dz1, x)
grad_b1 = dz1
print(grad_W1.round(4))

The whole second column is zero, and that is not a mistake. The second input was 0, so those weights were multiplied by nothing on this example and changing them could not have altered the answer. A weight that had no influence gets no blame. Figure 7.2 follows the full journey, forwards and then backwards.

> 🖼️ **Figure 7.2:** One example through the network and back. The forward pass computes the answer, and the backward pass carries the error signal from the loss to every weight.

You never have to trust these formulas, because you can check them with the nudging method from Chapter 6. The next cell measures the slope for the first weight by changing it slightly in each direction and watching the loss.

In [ ]:
def loss_with(W1):
    y_ = sigmoid(W2 @ sigmoid(W1 @ x + b1) + b2)
    return (y_ - target) ** 2

step = 1e-6
up, down = W1.copy(), W1.copy()
up[0, 0] += step
down[0, 0] -= step
measured = (loss_with(up) - loss_with(down)) / (2 * step)
print(round(measured, 6), round(grad_W1[0, 0], 6))

The measurement and the backward pass agree to six decimal places. **Backprop sends each neuron's error signal back along the weights that carried its influence forward, so every weight learns how much of the mistake belongs to it.**

## Training a Tiny Network by Hand

You now have both halves. Backprop gives the gradient for every weight, and Chapter 6 says to step opposite to it. Put them in a loop and the network trains itself.

The test is the puzzle from Chapter 3. There you built a network that solved XOR, but you chose its weights yourself by reasoning about what each neuron should do. This time nobody chooses anything. The next cell sets up all four XOR examples and starts the same small network from random weights.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
T = np.array([0, 1, 1, 0], dtype=float)

rng = np.random.default_rng(3)
W1 = rng.normal(0, 1, (2, 2))
b1 = np.zeros(2)
W2 = rng.normal(0, 1, 2)
b2 = 0.0

The training loop is the forward pass, the backward pass and the downhill step, repeated. It is the same arithmetic as before, written for all four examples at once, with the gradients averaged over them.

In [ ]:
history = []
for epoch in range(4001):
    H = sigmoid(X @ W1.T + b1)
    Y = sigmoid(H @ W2 + b2)
    history.append(((Y - T) ** 2).mean())

    dZ2 = 2 * (Y - T) * Y * (1 - Y) / len(X)
    dZ1 = np.outer(dZ2, W2) * H * (1 - H)

    W2 -= 0.5 * (dZ2 @ H)
    b2 -= 0.5 * dZ2.sum()
    W1 -= 0.5 * (dZ1.T @ X)
    b1 -= 0.5 * dZ1.sum(axis=0)

That is a complete neural network trainer in a dozen lines, and every line comes from something you have already derived. The next cell shows the loss as training goes on.

In [ ]:
for epoch in [0, 500, 1000, 2000, 4000]:
    print(epoch, round(history[epoch], 4))

The loss starts near 0.26, which Chapter 5 would tell you is roughly what guessing achieves, and ends at 0.002. The last cell asks the trained network for its four answers.

In [ ]:
print(sigmoid(sigmoid(X @ W1.T + b1) @ W2 + b2).round(2))

Those are the XOR answers. The network found its own weights for the puzzle that ended the first era of neural network research, starting from random numbers and using nothing but the slope of its own mistakes. Figure 7.3 shows the whole run.

> 🖼️ **Figure 7.3:** Training the tiny network on XOR. The loss barely moves for several hundred epochs while the hidden neurons sort themselves out, and then falls away quickly.

The shape of that curve is worth noticing, because it is not the smooth slide of Chapter 6. The loss hardly moves for the first few hundred epochs, then drops steeply, then flattens. The long flat start is the network's two hidden neurons arranging themselves into the "either" and "not both" roles you built by hand in Chapter 3, and until that arrangement appears there is little the output neuron can do.

One honest warning belongs here. This run used the random seed 3, and not every starting point works. Running the same code with other seeds, two out of five runs got stuck with a loss near 0.13, where the network answers correctly for some inputs and sits at 0.5 for others. That is Chapter 6's local minimum, met in the wild on the smallest network in this book. **Backprop finds a way down from wherever it starts, and where it starts can decide whether it arrives somewhere good.**

## Why Backprop Scales to Millions of Weights

Backprop is not the only way to get gradients. Chapter 6 measured a slope by nudging, and that works for any network at all, so it is fair to ask why anyone needs something cleverer.

The answer is counting. Nudging measures one weight at a time, and each measurement needs two forward passes, one up and one down. Backprop computes every gradient in a single backward sweep, and that sweep costs about as much as one forward pass, because it visits each connection once more. So the nudging method costs two passes per weight, while backprop costs about two passes in total, no matter how many weights there are.

In [ ]:
for n_in, n_hidden in [(2, 2), (20, 20), (784, 128)]:
    weights = n_hidden * n_in + n_hidden + n_hidden + 1
    print(weights, 2 * weights, 2)

Table 7.1 lays out what those numbers mean. For the tiny XOR network the saving hardly matters. For a modest network reading small images, nudging needs over two hundred thousand forward passes for a single step of gradient descent, while backprop needs the equivalent of two. Multiply that by the thousands of steps in a real training run, and the difference is between an afternoon and several years.

**Table 7.1: The cost of one step of gradient descent, counted in forward passes. Nudging grows with the number of weights, while backprop does not.**

| Network | Weights and biases | Passes by nudging | Passes by backprop |
|---|---|---|---|
| XOR, 2 inputs, 2 hidden | 9 | 18 | 2 |
| Small table, 20 inputs, 20 hidden | 441 | 882 | 2 |
| Small images, 784 inputs, 128 hidden | 100,609 | 201,218 | 2 |

**Backprop costs about one extra pass through the network no matter how many weights it has, which is the only reason training a network with millions of weights is possible at all.**

The reason it can be so cheap is the structure of the chain rule. Walking backwards, each layer computes its error signals from the layer above it, and those signals are then reused by every weight in that layer. Nudging throws all of that sharing away and recomputes the whole network from scratch for each individual weight.

To be precise, a backward sweep is not quite free. It usually costs somewhat more than a forward pass, often around twice as much, because each connection has to carry a signal back and then work out its own gradient from it. So a full training step is nearer three forward passes than two. That correction hardly matters, because the number stays the same whether the network has nine weights or nine million, while the nudging column of the table keeps growing forever.

There is one more thing to notice about the training loop you wrote. Deriving those gradient lines took a page of careful reasoning for a network with nine weights, and doing the same by hand for a network with a hundred layers is not something anyone should attempt. In Chapter 9 you will meet a tool that performs this exact calculation automatically for any network you can write down, and you will check its answers against the ones you worked out here.

## Practice and Review

Reading about deep learning is only half of learning it. Pulling an idea back out of your memory makes it stick much better than reading it again, so this section gives you two kinds of practice, one for your hands and one for your memory.

### Try It Yourself

1. **Check another gradient.** Use the nudging method on `W2[0]` and on `b2`, and check both against `grad_W2[0]` and `grad_b2`.
2. **Find a run that gets stuck.** Train the XOR network again with `default_rng(1)` instead of 3, print the final loss and the four answers, and say which inputs it still gets wrong.
3. **Feed it the other input.** Change `x` to `[0.0, 1.0]` in the single example section and recompute `grad_W1`. Work out which column of the gradient is zero this time, and why.

### Check Your Understanding

Try to answer each question from memory before you look at the answers that follow.

1. **What does the chain rule say about effects passing along a chain?**
2. **How does a hidden neuron find out how much of the mistake is its fault?**
3. **Why was one column of the first layer's gradient entirely zero?**
4. **Why does the XOR loss stay flat for hundreds of epochs before falling?**
5. **Why is backprop so much cheaper than measuring every slope by nudging?**

### Answers

1. That you multiply the effects of all the links to get the effect from one end to the other.
2. The neuron above it passes back its error signal, multiplied by the weight joining them, so a larger weight carries more blame.
3. Because that input was zero on this example, so those weights had no influence on the answer and no share of the blame.
4. Because the hidden neurons must first settle into useful roles, and until they do the output neuron has little to work with.
5. Because one backward sweep produces every gradient at once and reuses each layer's error signals, while nudging recomputes the whole network twice for every single weight.

## What Comes Next

You have now built a complete neural network from nothing. You know what a neuron is, how layers stack, why they need a bend, how to measure a mistake, how to walk downhill and how to find the slope for every weight at once. That is the entire machinery of deep learning, and from here it only gets bigger, never different.

- **Chapter 8, Tensors in PyTorch,** introduces the framework and the array type that everything in modern deep learning is built on.
- **Chapter 9, Autograd: Calculus for Free,** does this chapter's backward pass for you, and you will test it against the numbers you worked out by hand.

Part 3 begins with a fair question. If you can already write a network in numpy, as you just did, why learn a framework at all?